# RAG Document Loaders and Text Splitters Demo

This notebook demonstrates document-loading examples for plain text, PDF, CSV, and a webpage, then splits the webpage content into overlapping chunks. These are document-preparation building blocks commonly used before indexing in a Retrieval-Augmented Generation (RAG) pipeline; this notebook does not implement retrieval or answer generation.

The examples are intentionally simple and readable so they are easy to follow during portfolio review, interviews, or self-learning. The focus is on understanding how documents are ingested and prepared before they are used in a RAG system.

In [1]:
from dotenv import load_dotenv

# Load environment variables needed by the notebook session.
load_dotenv()

True

In [2]:
# Load a plain-text file to demonstrate the simplest document ingestion pattern.
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader("../data/speech.txt", encoding="utf8")
loaded_text_documents = text_loader.load()

C:\Users\vishe\AppData\Local\Temp\ipykernel_12124\1194905035.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
# Load a PDF document to show how resumes and reports can be used in a RAG pipeline.
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("../data/Vishesh_Mehta_Applied_Software_Engineer.pdf")
loaded_resume_documents = pdf_loader.load()

In [4]:
# Load structured tabular data from a CSV file and inspect how many records are available.
from langchain_community.document_loaders import CSVLoader

csv_loader = CSVLoader("../data/people.csv")
loaded_csv_documents = csv_loader.load()
print("Number of rows in the CSV file:", len(loaded_csv_documents))

Number of rows in the CSV file: 10


In [5]:
# View a sample CSV row to understand how the loader represents tabular content.
sample_csv_document = loaded_csv_documents[3]
print(sample_csv_document.page_content)

id: 4
name: Emma Chen
age: 27
city: Perth
last_login: 2026-09-25 07:44:21
purchase_count: 0
account_status: suspended


In [6]:
# Load content from a live webpage to demonstrate ingestion from online sources.
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader("https://medium.com/@unicodeveloper/the-ultimate-guide-to-jev-the-new-frontier-ai-for-faster-decisions-acd78e5f4c56")
loaded_web_documents = web_loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
# Inspect the metadata returned by the web loader for a source document.
loaded_web_documents[0].metadata

{'source': 'https://medium.com/@unicodeveloper/the-ultimate-guide-to-jev-the-new-frontier-ai-for-faster-decisions-acd78e5f4c56',
 'title': 'Medium',
 'description': 'The Ultimate Guide to Jev: The new Frontier AI for faster decisions ChatGPT co-creator spent two years building Jev. Here is what it does, when to use it, when not to and what people have already …',
 'language': 'en'}

In [8]:
# View the actual page content to see how the browser-based source is converted to text.
print(loaded_web_documents[0].page_content)

MediumThe Ultimate Guide to Jev: The new Frontier AI for faster decisions | by unicodeveloper | Sep, 2026 | MediumSitemapOpen in appSign upSign inMedium LogoGet appWriteSearchSign upSign inWhat Exactly does Jev Solve?What is a System One model?Jev vs a frontier LLMHow do you use Jev?Why calibrated confidence changes how you design systemsWhen should you use Jev instead of an LLM?The architecture that pays: the cascadePractical Solutions actually built with Jev1. 1,018 research papers classified for 8 cents2. A browser agent that books a flight in 7 seconds3. Computer use for a fiftieth of a cent per step4. A market maker that decides every blockchain block5. An autonomous drone with judgment at 2.5Hz6. Super Mario Bros from RAM, not pixelsAlso worth a lookOne Important Thing to Note. Take this Seriously!Frequently asked questionsJevArtificial IntelligenceFrontier TechValyuDeep ResearchThe Ultimate Guide to Jev: The new Frontier AI for faster decisionsunicodeveloper12 min read·Sep 17, 2

In [9]:
# Compare document counts, titles, sources, and text lengths before processing.
print("Documents loaded:", len(loaded_web_documents))
web_document_lengths = [len(document.page_content) for document in loaded_web_documents]
for index, document in enumerate(loaded_web_documents, start=1):
    title = document.metadata.get("title", "(title unavailable)")
    source = document.metadata.get("source", "(source unavailable)")
    print(f"{index}. {title} | {web_document_lengths[index - 1]:,} characters | {source}")
if web_document_lengths:
    average_length = sum(web_document_lengths) / len(web_document_lengths)
    print(f"Character lengths: min={min(web_document_lengths):,}, average={average_length:,.0f}, max={max(web_document_lengths):,}")

Documents loaded: 1
1. Medium | 20,093 characters | https://medium.com/@unicodeveloper/the-ultimate-guide-to-jev-the-new-frontier-ai-for-faster-decisions-acd78e5f4c56
Character lengths: min=20,093, average=20,093, max=20,093


In [10]:
# Inspect available metadata and bounded previews from the first result.
if loaded_web_documents:
    sample_web_document = loaded_web_documents[0]
    print("Metadata fields:", sorted(sample_web_document.metadata))
    print("Summary preview:", sample_web_document.metadata.get("summary", "(summary unavailable)")[:500])
    print("Content preview:", sample_web_document.page_content[:500])
else:
    print("No web documents were returned.")

Metadata fields: ['description', 'language', 'source', 'title']
Summary preview: (summary unavailable)
Content preview: MediumThe Ultimate Guide to Jev: The new Frontier AI for faster decisions | by unicodeveloper | Sep, 2026 | MediumSitemapOpen in appSign upSign inMedium LogoGet appWriteSearchSign upSign inWhat Exactly does Jev Solve?What is a System One model?Jev vs a frontier LLMHow do you use Jev?Why calibrated confidence changes how you design systemsWhen should you use Jev instead of an LLM?The architecture that pays: the cascadePractical Solutions actually built with Jev1. 1,018 research papers classified 


In [11]:
# Configure overlapping character chunks; the next cell applies this splitter to the loaded webpage.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [12]:
data_chunks = text_splitter.split_documents(loaded_web_documents)
print("Number of chunks created from the web document:", len(data_chunks))
print(data_chunks[2].page_content)

Number of chunks created from the web document: 25
stealth!Press enter or click to view image in full sizeWhat Exactly does Jev Solve?You have a model that understands language and need a decision your code can branch on.Ask the model for language, then you write a second program whose entire job is to claw a decision back out of the language. You prompt it to “respond only with valid JSON.” You parse. You validate. You catch the case where it wrapped the object in a markdown fence. You retry when the enum came back as "technical support" instead of technical. You write a fallback for when it apologises instead of answering.Press enter or click to view image in full sizeThat second program is not a small thing. In most production AI systems it is a meaningful share of the code, and nearly all of the flakiness.Diogo Almeida (the founder) said this:,“At OpenAI, I helped build the methods that made language models useful at following instructions and talking with people. That work ended u